# Lesson 8 — Distilling networks into equations (symbolic regression)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karefyllidis/PlugFlowML/blob/main/notebooks/Main_8_symbolic_regression_SR.ipynb)

*Part of the [PlugFlowML course](../README.md): machine-learning surrogates for physical simulation, taught on a chemical reactor.*

**Mode:** hands-on (Colab-ready) · **Runtime:** ~15 min on free Colab (plus PySR's one-off Julia setup on first run) · **Builds on:** Lessons 6–7

**What you'll learn**

1. Distil a trained neural network into closed-form equations by treating it as a cheap data generator (knowledge distillation with a symbolic-regression student).
2. Run PySR's evolutionary search and read its accuracy-vs-complexity Pareto front, instead of blindly taking the most accurate equation.
3. Validate a distilled model twice — against its teacher (fidelity) and against the original simulation data (accuracy).
4. Export the equations as a dependency-free Python module, ready for fast optimisation (Lesson 10) or embedding outside Python.

**The example system.** The teacher network predicts full axial profiles of a plug-flow reactor — temperature, pressure, velocity, density and lumped species mass fractions — from six inlet conditions plus the axial position z/L (see [Lesson 1](Main_1_run_pfr.ipynb) for the reactor itself). This lesson replaces that network with one explicit equation per output. The default teacher is Lesson 6's SimpleNN (`teacher_stem` in `configs/ml/main8_symbolic_regression_config.json`); set it to `'pinn_pfr'` to distil Lesson 7's PINN instead — that variant is the one Lesson 9 validates.

In [ ]:
# ══ 0. COLAB BOOTSTRAP ══
# Running locally? This cell is a no-op — just run it and move on.
import sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    if Path.cwd().name != "notebooks":            # fresh Colab VM
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/karefyllidis/PlugFlowML.git"], check=True)
        %cd PlugFlowML/notebooks
    %pip install -q pysr
    _REL = "https://github.com/karefyllidis/PlugFlowML/releases/download/sample-data-v1"
    import urllib.request, zipfile
    for _f, _d in [("features_targets_training_data_complete_sample150.pkl", "../data/processed"),
                   ("models_pretrained_sample.zip", "../models")]:
        _p = Path(_d); _p.mkdir(parents=True, exist_ok=True)
        if not (_p / _f).exists():
            print(f"Downloading {_f} …")
            urllib.request.urlretrieve(f"{_REL}/{_f}", _p / _f)
            if _f.endswith(".zip"):
                with zipfile.ZipFile(_p / _f) as _z:
                    _z.extractall(_p)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 1. SETUP & IMPORTS
# ════════════════════════════════════════════════════════════════════════════
import os
import sys
import json
import joblib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import r2_score, mean_absolute_error

try:
    from pysr import PySRRegressor
    PYSR_AVAILABLE = True
except ImportError:
    PYSR_AVAILABLE = False
    print('[warn] PySR not installed. Run: pip install pysr')

PROJECT_ROOT = Path(os.getcwd())
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models import SimpleNN, PINNPFR
from src.utils.plot_style import setup_matplotlib
setup_matplotlib()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 2. PATHS & FLAGS
# ════════════════════════════════════════════════════════════════════════════
# Edit configs/ml/main8_symbolic_regression_config.json to change these
# (edit and re-run this cell; no kernel restart needed).
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'ml' / 'main8_symbolic_regression_config.json'
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        _cfg = json.load(f)
else:
    _cfg = {}
    print(f'[WARN] Config not found: {CONFIG_PATH}. Using defaults.')

IF_PLOT_SHOWN  = _cfg.get('if_plot_shown', True)
IF_PLOT_EXPORT = _cfg.get('if_plot_export', True)
IF_SR_EXPORT   = _cfg.get('if_sr_export', True)

# PySR budget — increase for production runs
SR_N_ITERATIONS  = _cfg.get('sr_n_iterations', 100)
SR_POPULATIONS   = _cfg.get('sr_populations', 30)
SR_POPULATION_SIZE = _cfg.get('sr_population_size', 33)
SR_MAXSIZE       = _cfg.get('sr_maxsize', 25)
SR_PROCS         = _cfg.get('sr_procs', 0)

N_DISTILL_SAMPLES = _cfg.get('n_distill_samples', 5_000)

RANDOM_STATE = _cfg.get('random_state', 42)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# ── Teacher selection ──────────────────────────────────────────────────────
# Options:
#   'simple_nn_full_profile'  → Main_6 SimpleNN, full axial profile (6+z inputs)
#   'pinn_pfr'                → Main_7 PINNPFR,  full axial profile (6+z inputs)
TEACHER_STEM = _cfg.get('teacher_stem', 'simple_nn_full_profile')

_stem_to_export = {
    'simple_nn_full_profile': ('sr_full_profile', 'sr_full_profile'),
    'pinn_pfr':               ('sr_pinn',         'sr_pinn'),
}
_export_subdir, EXPORT_STEM = _stem_to_export.get(TEACHER_STEM, ('sr_custom', 'sr_custom'))
IS_PROFILE_MODEL = TEACHER_STEM in ('simple_nn_full_profile', 'pinn_pfr')

MODELS_DIR = PROJECT_ROOT / 'models'
FIGS_DIR   = PROJECT_ROOT / 'outputs' / 'figures' / 'Main_8_symbolic_regression'
EXPORT_DIR = PROJECT_ROOT / 'models' / _export_subdir
FIGS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Teacher stem:    {TEACHER_STEM}')
print(f'Profile model:   {IS_PROFILE_MODEL}')
print(f'Export dir:      {EXPORT_DIR}')

> 🧠 **Concept — Knowledge distillation.** A trained network is more than a predictor — it is a cheap, effectively infinite data generator. The original simulator takes minutes per reactor run; the teacher network answers any input in microseconds, so a *student* model can be trained on as many (input, output) pairs as it likes, sampled wherever they are most informative. Distillation transfers the teacher's learned function into a different model class — here, closed-form equations — trading a little accuracy for properties the network lacks: speed, portability, interpretability. The catch is that the student learns the teacher's function, mistakes included, which is why Lesson 9 re-validates everything against the original simulation.
>
> *In your domain:* the same trick turns a heavyweight climate or CFD emulator into pocket-sized response formulas for controllers and design screens.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 3. LOAD TEACHER MODEL
# ════════════════════════════════════════════════════════════════════════════
manifest_path = MODELS_DIR / TEACHER_STEM / f'{TEACHER_STEM}_manifest.json'
_source_nb = {'simple_nn_full_profile': 'Main_6', 'pinn_pfr': 'Main_7'}
if not manifest_path.exists():
    raise FileNotFoundError(
        f'Teacher artifacts not found: {manifest_path} — run the Colab bootstrap cell '
        f'(it downloads the pretrained models), or train your own teacher in Lesson 6/7 '
        f'({_source_nb.get(TEACHER_STEM, "source notebook")} with IF_MODEL_EXPORT=True).'
    )

with open(manifest_path) as f:
    manifest = json.load(f)

arch        = manifest['architecture']
inlet_cols  = manifest.get('inlet_cols') or manifest.get('feature_cols', [])
target_cols = manifest['target_cols']
print(f'Teacher stem:   {TEACHER_STEM}')
print(f'Trained at:     {manifest.get("run_at", "unknown")}')
print(f'Inputs  ({len(inlet_cols)}): {inlet_cols}')
print(f'Outputs ({len(target_cols)}): {target_cols}')

scalers  = joblib.load(MODELS_DIR / TEACHER_STEM / f'{TEACHER_STEM}_scalers.joblib')
scaler_X = scalers['scaler_X']
scaler_y = scalers.get('scaler_y')

# Instantiate the correct model class
if TEACHER_STEM == 'pinn_pfr':
    teacher = PINNPFR(
        arch['in_features'], arch['h1'], arch['h2'], arch['h3'],
        arch['out_features'], dropout=arch['dropout']
    )
else:
    teacher = SimpleNN(
        arch['in_features'], arch['h1'], arch['h2'], arch['h3'],
        arch['out_features'], dropout=arch['dropout']
    )

teacher.load_state_dict(torch.load(
    MODELS_DIR / TEACHER_STEM / f'{TEACHER_STEM}_state_dict.pt',
    map_location='cpu', weights_only=True,
))
teacher.eval()
print(f'Loaded {teacher.__class__.__name__}: {sum(p.numel() for p in teacher.parameters()):,} params')

> 🧠 **Concept — Coordinates as inputs: equations for whole profiles.** The teacher takes the six inlet conditions *plus the axial position* (z and z/L) as inputs, so every distillation sample below is one (run, position) pair, not one experiment. The equations PySR finds inherit z/L as a variable: each is a formula for the entire reactor profile — evaluate it at z/L = 1 for the exit plane, or sweep z/L from 0 to 1 to trace the whole curve. Without the coordinate input you would need a separate set of equations per axial station, and interpolating between stations would be your problem. Watch for `relative_position` appearing inside the discovered expressions — when it does, the equation is genuinely describing how the reactor state evolves along its length.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 4. GENERATE DISTILLATION DATASET
# ════════════════════════════════════════════════════════════════════════════
# Prefer the config-pinned dataset (course default: the sample150 release file);
# fall back to the newest features_targets_*.pkl if no stem is pinned.
_stem = _cfg.get('processed_stem')
if _stem:
    data_file = PROJECT_ROOT / 'data' / 'processed' / f'features_targets_training_data_complete_{_stem}.pkl'
    if not data_file.exists():
        raise FileNotFoundError(
            f'{data_file.name} not found in data/processed — run the Colab bootstrap cell '
            f'(it downloads the sample dataset) or export it in Lesson 3 (Main_3).'
        )
else:
    data_files = sorted((PROJECT_ROOT / 'data' / 'processed').glob('features_targets_*.pkl'))
    if not data_files:
        raise FileNotFoundError(
            'No features_targets_*.pkl in data/processed — run the Colab bootstrap cell '
            'or Lesson 3 (Main_3).'
        )
    data_file = data_files[-1]

_loaded = pd.read_pickle(data_file)
if isinstance(_loaded, dict):
    # Main_3 portable payload: {'df_features': ..., 'df_target': ..., ...}
    df_all = pd.concat([_loaded['df_features'], _loaded['df_target']], axis=1)
    df_all = df_all.loc[:, ~df_all.columns.duplicated()]
else:
    df_all = _loaded  # legacy single-DataFrame pickle
print(f'Loaded {len(df_all):,} rows from {data_file.name}')

avail_inlet = [c for c in inlet_cols if c in df_all.columns]
if len(avail_inlet) < len(inlet_cols):
    missing = set(inlet_cols) - set(avail_inlet)
    raise KeyError(f'Feature columns missing from processed data: {missing}')

# Profile models include z_position_m / relative_position as inputs.
# Sample from full dataset rows (each row is a unique (run, z) pair).
X_pool = df_all[inlet_cols].dropna().values
print(f'Profile mode: sampling from {len(X_pool):,} (run, z) rows')

replace = len(X_pool) < N_DISTILL_SAMPLES
idx = np.random.choice(len(X_pool), size=N_DISTILL_SAMPLES, replace=replace)
X_distill = X_pool[idx]

X_s = scaler_X.transform(X_distill)
with torch.no_grad():
    y_s = teacher(torch.tensor(X_s, dtype=torch.float32)).numpy()
y_distill = scaler_y.inverse_transform(y_s) if scaler_y is not None else y_s

df_distill = pd.DataFrame(X_distill, columns=inlet_cols)
df_distill[target_cols] = y_distill
print(f'Distillation dataset: {df_distill.shape}')
df_distill.describe().round(3)

> 🧠 **Concept — Symbolic regression.** Most regressors fix a functional form and fit its parameters; symbolic regression searches over the *forms themselves* — expression trees whose internal nodes are operators (`+`, `×`, `exp`, …) and whose leaves are variables and constants. PySR explores this space with an evolutionary algorithm: populations of candidate expressions mutate and recombine, survivors are selected on fit quality, and the constants inside each candidate are refined numerically. Because the space is discrete and enormous, the search is stochastic — re-runs can land on different equations. The result is not one equation but a **Pareto front**: the best expression found at each complexity level. You rarely want the most accurate member — it is usually the longest, hardest to interpret, and most prone to fitting the teacher's quirks. Look for the "knee", where extra complexity stops buying accuracy.

> ⏳ **Note — PySR compiles a Julia backend on first run.** PySR's search engine is written in Julia. The first time you call `fit()` on a fresh machine (every new Colab VM counts), it downloads and precompiles that backend — expect several minutes with little visible progress before the search starts. This is normal and happens once per machine; subsequent fits start in seconds.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 5. SYMBOLIC REGRESSION WITH PYSR
# ════════════════════════════════════════════════════════════════════════════
# We run one multi-output PySR fit for all targets simultaneously.
# PySR finds one equation per output independently.

if not PYSR_AVAILABLE:
    raise RuntimeError('Install PySR: pip install pysr')

X_sr = df_distill[inlet_cols].values
y_sr = df_distill[target_cols].values

sr_model = PySRRegressor(
    niterations     = SR_N_ITERATIONS,
    populations     = SR_POPULATIONS,
    population_size = SR_POPULATION_SIZE,
    maxsize         = SR_MAXSIZE,
    binary_operators = ['+', '-', '*', '/'],
    unary_operators  = ['exp', 'sqrt', 'square'],
    parsimony        = 0.01,
    procs            = SR_PROCS,
    random_state     = RANDOM_STATE,
    verbosity        = 1,
    # output_jax_format=True,  # enable for JAX export
)

print(f'Running PySR on {len(X_sr):,} samples × {len(target_cols)} outputs ...')
sr_model.fit(X_sr, y_sr, variable_names=inlet_cols)
print('PySR done.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 5b. BEST EQUATIONS TABLE
# ════════════════════════════════════════════════════════════════════════════
equations = []
for i, col in enumerate(target_cols):
    best = sr_model.get_best()[i]
    equations.append({
        'target':    col,
        'equation':  str(best['equation']),
        'latex':     sr_model.latex()[i] if hasattr(sr_model, 'latex') else '',
        'loss':      float(best['loss']),
        'complexity': int(best['complexity']),
    })

df_eqs = pd.DataFrame(equations)
print(df_eqs[['target', 'complexity', 'loss', 'equation']].to_string(index=False))

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6. EVALUATE: SR vs TEACHER NN
# ════════════════════════════════════════════════════════════════════════════
y_sr_pred = sr_model.predict(X_sr)  # (N, n_outputs)

metrics = []
for i, col in enumerate(target_cols):
    yt = y_distill[:, i]
    yp = y_sr_pred[:, i] if y_sr_pred.ndim == 2 else y_sr_pred
    r2  = r2_score(yt, yp)
    mae = mean_absolute_error(yt, yp)
    metrics.append({'target': col, 'R2_sr_vs_nn': r2, 'MAE_sr_vs_nn': mae})

df_metrics = pd.DataFrame(metrics)
print(df_metrics.to_string(index=False))
print(f'\nMean R²  (SR vs NN): {df_metrics.R2_sr_vs_nn.mean():.4f}')
print(f'Mean MAE (SR vs NN): {df_metrics.MAE_sr_vs_nn.mean():.4e}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 7. PARITY PLOTS: SR vs TEACHER NN
# ════════════════════════════════════════════════════════════════════════════
n_cols = 3
n_rows = int(np.ceil(len(target_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3.8 * n_rows), squeeze=False)
setup_matplotlib(axes)

for i, col in enumerate(target_cols):
    ax  = axes[i // n_cols][i % n_cols]
    yt  = y_distill[:, i]
    yp  = y_sr_pred[:, i] if y_sr_pred.ndim == 2 else y_sr_pred
    r2  = df_metrics.loc[df_metrics.target == col, 'R2_sr_vs_nn'].values[0]

    ax.scatter(yt, yp, s=4, alpha=0.3, c='b', rasterized=True)
    lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
    ax.plot(lims, lims, 'r-', lw=1)
    ax.set_title(f'{col}  R²={r2:.3f}', fontsize=9)
    ax.set_xlabel('NN teacher', fontsize=8)
    ax.set_ylabel('SR expression', fontsize=8)

for j in range(i + 1, n_rows * n_cols):
    axes[j // n_cols][j % n_cols].set_visible(False)

fig.suptitle('SR vs SimpleNN teacher (distillation parity)', fontsize=11)
plt.tight_layout()
if IF_PLOT_EXPORT:
    fig.savefig(FIGS_DIR / 'parity_sr_vs_nn.png', dpi=200, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 8. VALIDATE SR ON CANTERA TEST DATA
# ════════════════════════════════════════════════════════════════════════════
# Compare SR expressions directly against Cantera exit-plane ground truth
# (inlet conditions only).

# Load raw exit-plane rows (last axial point per run, or rows where z/L == 1)
avail_cols = [c for c in target_cols if c in df_all.columns]
inlet_avail = [c for c in inlet_cols if c in df_all.columns]

if avail_cols and inlet_avail:
    df_exit = df_all[df_all.get('relative_position', pd.Series(0.0, index=df_all.index)) >= 0.99]
    if len(df_exit) == 0:
        df_exit = df_all.groupby(inlet_avail).last().reset_index()

    X_test_c = df_exit[inlet_avail].values
    y_test_c = df_exit[avail_cols].values
    y_sr_test = sr_model.predict(X_test_c)

    ct_metrics = []
    for i, col in enumerate(avail_cols):
        yt = y_test_c[:, i]
        yp = y_sr_test[:, i] if y_sr_test.ndim == 2 else y_sr_test
        ct_metrics.append({'target': col,
                           'R2_sr_vs_cantera': r2_score(yt, yp),
                           'MAE_sr_vs_cantera': mean_absolute_error(yt, yp)})
    df_ct = pd.DataFrame(ct_metrics)
    print(df_ct.to_string(index=False))
    print(f'\nMean R² (SR vs Cantera): {df_ct.R2_sr_vs_cantera.mean():.4f}')
else:
    print('[skip] Cantera validation: target columns not in training data file.')
    df_ct = pd.DataFrame()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 8b. R² BAR CHART — SR fidelity
# ════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
setup_matplotlib(axes)

# Left: SR vs NN
ax = axes[0]
ax.barh(df_metrics['target'], df_metrics['R2_sr_vs_nn'],
        facecolor='white', edgecolor='b', hatch='///')
ax.axvline(1.0, color='k', lw=0.8, ls='--')
ax.axvline(0.0, color='r', lw=0.8, ls='--', label='naive baseline')
ax.set_xlabel('R²')
ax.set_title('SR fidelity vs NN teacher')
ax.legend(fontsize=8)

# Right: SR vs Cantera (if available)
ax = axes[1]
if not df_ct.empty:
    ax.barh(df_ct['target'], df_ct['R2_sr_vs_cantera'],
            facecolor='white', edgecolor='r', hatch='///')
    ax.axvline(1.0, color='k', lw=0.8, ls='--')
    ax.axvline(0.0, color='r', lw=0.8, ls='--')
    ax.set_xlabel('R²')
    ax.set_title('SR accuracy vs Cantera ground truth')
else:
    ax.set_visible(False)

plt.tight_layout()
if IF_PLOT_EXPORT:
    fig.savefig(FIGS_DIR / 'r2_bar_sr_fidelity.png', dpi=200, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

> 🧠 **Concept — What closed-form surrogates buy you.** An equation evaluates in microseconds using plain NumPy — no PyTorch, no GPU, no weight files; the model *is* the text. That makes it embeddable almost anywhere (a spreadsheet, a process simulator, a C routine inside a CFD solver), trivially version-controlled, and auditable by inspection: you can see whether temperature enters through an exponential, or whether a variable dropped out entirely. The price is accuracy — losing a few points of R² relative to the teacher is typical. Whether the trade is worth it depends on the downstream job: Lesson 10's Bayesian optimiser calls the surrogate thousands of times, which is exactly where microsecond inference pays for itself.
>
> *In your domain:* embedded controllers, real-time digital twins, and legacy Fortran codes all accept an equation far more readily than a neural-network runtime.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 9. DISPLAY EQUATIONS (LATEX)
# ════════════════════════════════════════════════════════════════════════════
from IPython.display import display, Math

for row in df_eqs.itertuples():
    print(f'--- {row.target} (complexity={row.complexity}, loss={row.loss:.4e}) ---')
    print(f'  Python: {row.equation}')
    if row.latex:
        display(Math(row.latex))
    print()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 10. EXPORT
# ════════════════════════════════════════════════════════════════════════════
if IF_SR_EXPORT:
    run_at = datetime.now().isoformat(timespec='seconds')

    export_meta = {
        'run_at':            run_at,
        'teacher_stem':      TEACHER_STEM,
        'teacher_class':     teacher.__class__.__name__,
        'is_profile_model':  IS_PROFILE_MODEL,
        'teacher_manifest':  str(manifest_path),
        'n_distill_samples': N_DISTILL_SAMPLES,
        'sr_params': {
            'niterations':    SR_N_ITERATIONS,
            'populations':    SR_POPULATIONS,
            'population_size': SR_POPULATION_SIZE,
            'maxsize':        SR_MAXSIZE,
        },
        'inlet_cols':  inlet_cols,
        'target_cols': target_cols,
        'equations':   equations,
        'metrics_sr_vs_nn': df_metrics.to_dict(orient='records'),
    }
    if not df_ct.empty:
        export_meta['metrics_sr_vs_cantera'] = df_ct.to_dict(orient='records')

    meta_path = EXPORT_DIR / f'{EXPORT_STEM}_manifest.json'
    with open(meta_path, 'w') as f:
        json.dump(export_meta, f, indent=2)
    print(f'[OK] manifest → {meta_path}')

    # Python module — one function per output
    py_path = EXPORT_DIR / f'{EXPORT_STEM}_equations.py'
    lines = [
        f'"""Auto-generated symbolic expressions (Main_8 PySR distillation, teacher={TEACHER_STEM})."""\n',
        'import numpy as np\n\n',
    ]
    for row in df_eqs.itertuples():
        fname = row.target.replace('.', '_').replace(' ', '_')
        args  = ', '.join(inlet_cols)
        lines.append(f'def predict_{fname}({args}):\n')
        lines.append(f'    return {row.equation}\n\n')
    with open(py_path, 'w') as f:
        f.writelines(lines)
    print(f'[OK] equations module → {py_path}')

    csv_path = EXPORT_DIR / f'{EXPORT_STEM}_metrics.csv'
    df_metrics.to_csv(csv_path, index=False)
    print(f'[OK] metrics → {csv_path}')

    print(f'\n=== Export summary (teacher: {TEACHER_STEM}) ===')
    print(f'  Mean R² (SR vs NN):      {df_metrics.R2_sr_vs_nn.mean():.4f}')
    if not df_ct.empty:
        print(f'  Mean R² (SR vs Cantera): {df_ct.R2_sr_vs_cantera.mean():.4f}')

---

#### 💪 Exercise 8.1 — Score one equation at the exit plane

The distillation metrics above average over the whole profile. Restrict the comparison to the **exit plane** (z/L ≥ 0.99) — the part of the reactor most downstream applications care about — and score the SR equation for a single target against the teacher there. Try `'temperature_K'` first, then a species such as `'Y_lump_chem_olefins'`. You should see exit-plane R² close to the full-profile value for smooth state variables, and typically somewhat worse for species with sharp axial gradients.

In [ ]:
# 💪 Exercise 8.1 — your turn (edit and re-run)
# TODO: change TARGET to another name from target_cols (e.g. 'Y_lump_chem_olefins').
TARGET = 'temperature_K' if 'temperature_K' in target_cols else target_cols[0]

_df_exit = df_all.copy()
if 'relative_position' in _df_exit.columns:
    _df_exit = _df_exit[_df_exit['relative_position'] >= 0.99]
    if len(_df_exit) == 0:
        _df_exit = df_all.copy()
_X_exit = _df_exit[inlet_cols].dropna().values

# Teacher prediction on the exit-plane rows
with torch.no_grad():
    _y_s = teacher(torch.tensor(scaler_X.transform(_X_exit), dtype=torch.float32)).numpy()
_y_teacher = scaler_y.inverse_transform(_y_s) if scaler_y is not None else _y_s

# SR prediction on the same rows
_y_sr = sr_model.predict(_X_exit)

_i  = target_cols.index(TARGET)
_yt = _y_teacher[:, _i]
_yp = _y_sr[:, _i] if _y_sr.ndim == 2 else _y_sr
print(f'{TARGET} @ exit plane ({len(_X_exit)} rows):')
print(f'  R²  (SR vs teacher): {r2_score(_yt, _yp):.4f}')
print(f'  MAE (SR vs teacher): {mean_absolute_error(_yt, _yp):.4e}')
print(f'  full-profile R² for comparison: '
      f"{df_metrics.loc[df_metrics.target == TARGET, 'R2_sr_vs_nn'].values[0]:.4f}")

<details><summary><b>💡 Solution & discussion — Exercise 8.1</b></summary>

```python
TARGET = 'Y_lump_chem_olefins' if 'Y_lump_chem_olefins' in target_cols else target_cols[-1]

_df_exit = df_all.copy()
if 'relative_position' in _df_exit.columns:
    _df_exit = _df_exit[_df_exit['relative_position'] >= 0.99]
    if len(_df_exit) == 0:
        _df_exit = df_all.copy()
_X_exit = _df_exit[inlet_cols].dropna().values

with torch.no_grad():
    _y_s = teacher(torch.tensor(scaler_X.transform(_X_exit), dtype=torch.float32)).numpy()
_y_teacher = scaler_y.inverse_transform(_y_s) if scaler_y is not None else _y_s
_y_sr = sr_model.predict(_X_exit)

_i  = target_cols.index(TARGET)
_yt = _y_teacher[:, _i]
_yp = _y_sr[:, _i] if _y_sr.ndim == 2 else _y_sr
print(f'{TARGET} @ exit plane: R²={r2_score(_yt, _yp):.4f}, MAE={mean_absolute_error(_yt, _yp):.4e}')
```

Restricting to the exit plane removes the easy points: near the inlet every model predicts values close to the inlet state, which flatters full-profile R². For smooth state variables like temperature the equation usually holds up at the exit; for species, the exit composition is the accumulation of all chemistry along z, so an equation that shadows the teacher well on average can lose several points of R² exactly where downstream users need it most — Lesson 10's optimiser targets exit-plane olefins.
</details>

#### 💪 Exercise 8.2 — Walk the Pareto front

PySR kept every non-dominated equation it found, not just the winner. Print the full accuracy-vs-complexity front for one target, then pick a deliberately simpler equation (lower `MAX_COMPLEXITY`) and quantify what the simplicity costs in loss. You should find a "knee": below some complexity the loss deteriorates sharply, while above it extra terms buy almost nothing.

In [ ]:
# 💪 Exercise 8.2 — your turn (edit and re-run)
# TODO: change TARGET_IDX (which output) and MAX_COMPLEXITY (how simple an equation you accept).
TARGET_IDX     = 0
MAX_COMPLEXITY = 8

_eqs   = sr_model.equations_
pareto = (_eqs[TARGET_IDX] if isinstance(_eqs, list) else _eqs).copy()
print(f'Pareto front — {target_cols[TARGET_IDX]}:')
print(pareto[['complexity', 'loss', 'equation']].to_string(index=False))

_best = pareto.loc[pareto['loss'].idxmin()]
_cand = pareto[pareto['complexity'] <= MAX_COMPLEXITY]
_simpler = _cand.loc[_cand['loss'].idxmin()] if len(_cand) else _best

print(f"\nMost accurate: complexity={_best['complexity']}, loss={_best['loss']:.4e}")
print(f"Simpler pick : complexity={_simpler['complexity']}, loss={_simpler['loss']:.4e}")
print(f"Accuracy cost (loss ratio, simpler/best): {_simpler['loss'] / max(_best['loss'], 1e-300):.2f}x")
print(f"Simpler equation: {_simpler['equation']}")

<details><summary><b>💡 Solution & discussion — Exercise 8.2</b></summary>

```python
TARGET_IDX = 0
_eqs = sr_model.equations_
pareto = (_eqs[TARGET_IDX] if isinstance(_eqs, list) else _eqs)
_best = pareto.loc[pareto['loss'].idxmin()]
for MAX_COMPLEXITY in (4, 8, 12, 16):
    _cand = pareto[pareto['complexity'] <= MAX_COMPLEXITY]
    if len(_cand) == 0:
        continue
    _simpler = _cand.loc[_cand['loss'].idxmin()]
    print(f"cap {MAX_COMPLEXITY:>2}: complexity={_simpler['complexity']:>2}, "
          f"loss={_simpler['loss']:.3e}  ({_simpler['loss'] / max(_best['loss'], 1e-300):.1f}x best)")
```

Sweeping the cap makes the knee explicit: loss typically falls steeply up to some complexity and then flattens — beyond the knee, additional terms mostly fit the teacher's idiosyncrasies rather than the underlying trend. PySR's `score` column formalises this (it rewards loss reduction per unit of added complexity). Choosing at the knee also guards against a subtle failure mode: very long equations can contain near-cancelling terms that behave wildly outside the sampled domain, even when their in-sample loss is lowest.
</details>

#### 💪 Exercise 8.3 — Trace a whole profile with one equation

The discovered equations take z/L as an input, so a single formula describes the entire axial profile. Pick one run from the dataset, sweep its rows from inlet to exit, and overlay Cantera (ground truth), the NN teacher, and the SR equation. Try a different `RUN_IDX` and a species target. Expect the SR curve to track the teacher closely — and to inherit the teacher's deviations from Cantera rather than fixing them.

In [ ]:
# 💪 Exercise 8.3 — your turn (edit and re-run)
# TODO: change RUN_IDX to trace a different run, and TARGET to another output.
RUN_IDX = 0
TARGET  = 'temperature_K' if 'temperature_K' in target_cols else target_cols[0]

_run_cols = [c for c in inlet_cols if c not in ('z_position_m', 'relative_position')]
_run_ids  = df_all.groupby(_run_cols, dropna=False).ngroup()
_run      = RUN_IDX % _run_ids.nunique()
_rows     = df_all[_run_ids == _run].dropna(subset=inlet_cols)
if 'relative_position' in _rows.columns:
    _rows = _rows.sort_values('relative_position')

_X = _rows[inlet_cols].values
with torch.no_grad():
    _y_s = teacher(torch.tensor(scaler_X.transform(_X), dtype=torch.float32)).numpy()
_y_teacher = scaler_y.inverse_transform(_y_s) if scaler_y is not None else _y_s
_y_sr = sr_model.predict(_X)

_i = target_cols.index(TARGET)
_z = _rows['relative_position'].values if 'relative_position' in _rows.columns else np.arange(len(_rows))

fig, ax = plt.subplots(figsize=(5.5, 3.6))
setup_matplotlib(ax)
if TARGET in _rows.columns:
    ax.plot(_z, _rows[TARGET].values, 'k-', lw=1.8, label='Cantera')
ax.plot(_z, _y_teacher[:, _i], 'b--', lw=1.4, label='NN teacher')
ax.plot(_z, _y_sr[:, _i] if _y_sr.ndim == 2 else _y_sr, 'r-.', lw=1.4, label='SR equation')
ax.set_xlabel('z/L')
ax.set_ylabel(TARGET)
ax.set_title(f'{TARGET} — run {_run} (one equation, whole profile)', fontsize=9)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

<details><summary><b>💡 Solution & discussion — Exercise 8.3</b></summary>

```python
RUN_IDX = 3
TARGET  = 'Y_lump_chem_olefins' if 'Y_lump_chem_olefins' in target_cols else target_cols[-1]
# ...then re-run the scaffold above unchanged.
```

A single closed-form expression traces the whole axial curve because z/L is just another input variable to it. Two things are worth noticing. First, the SR curve follows the *teacher*, not Cantera: where the network deviates from the simulation, the equation deviates with it — distillation cannot repair its teacher. Second, smoothness: the equation is infinitely differentiable in z/L, so it interpolates smoothly between the axial grid points the teacher was trained on — convenient, but not a guarantee of physical behaviour between them.
</details>

---

## Summary

- **Teacher**: set by `TEACHER_STEM` — supports `SimpleNN` (Main_6) and `PINNPFR` (Main_7).
- **Student**: PySR symbolic expressions — one closed-form equation per output target.
- **Distillation**: teacher queried on N_DISTILL_SAMPLES inlet points sampled from the training data distribution. Profile models include z/L as an additional input.
- **Exports** (under `models/<sr_subdir>/`):
  - `*_manifest.json` — metadata + equations as strings
  - `*_equations.py` — callable Python functions (one per target, NumPy-compatible)
  - `*_metrics.csv` — R² and MAE vs teacher NN (and vs Cantera if available)

### Export stems by teacher
| `TEACHER_STEM` | Export directory | Equations file |
|---|---|---|
| `simple_nn_full_profile` | `models/sr_full_profile/` | `sr_full_profile_equations.py` |
| `pinn_pfr` | `models/sr_pinn/` | `sr_pinn_equations.py` |

### Operators
Binary: `+`, `-`, `*`, `/`. Unary: `exp`, `sqrt`, `square`.

### Tips for production
- Increase `SR_N_ITERATIONS` (≥ 500) and `N_DISTILL_SAMPLES` (≥ 20 000).
- Add `log` to `unary_operators` for species spanning decades.
- Use short `variable_names` aliases (`T_in`, `P_in`, …) for cleaner LaTeX.